# FDS Pokémon — reproduce the three final submissions

This notebook clones the GitHub repository and runs each approach in a fresh Python process.
The model code lives in the three matching folders under `submissions/`.

**Current status: working draft, not ready for final hand-in.** The V2 source is incomplete.
A previous fresh V5 run differed from its submitted CSV in 43 of 5,000 predictions.
Preflight stops before training if a selected approach is incomplete, and the final export
requires all three prediction fingerprints to match the original submissions.

For the final Kaggle version: attach the competition data, enable Internet for cloning/installing,
keep all three approaches selected, then restart the session and run all cells.

## Setup — clone the repository

`REPO_REF` selects the recorded code version. The checkout below is the only source of model code.
The repository must be public when the notebook is shared for the course.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get("FDS_REPO_URL", "https://github.com/teoang/fds-pokemon-challenge.git")
REPO_REF = os.environ.get("FDS_REPO_REF", "2a58de5570c9aca5916f8ae95662ea59eca52707")
WORKSPACE = Path(os.environ.get(
    "FDS_NOTEBOOK_WORK",
    "/kaggle/working" if Path("/kaggle/working").exists()
    else str(Path.cwd() / "outputs/local/notebook_workspace"),
)).resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPO = WORKSPACE / "fds-pokemon-code"
git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPO_URL, str(REPO)],
                   check=True, env=git_env)
else:
    if not (REPO / ".git").is_dir():
        raise RuntimeError(f"{REPO} exists and is not a Git clone")
    remote = subprocess.check_output(
        ["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    if remote != REPO_URL or dirty:
        raise RuntimeError("The existing checkout has another origin or local changes; use a new WORKSPACE")
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_REF],
               check=True, env=git_env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", "--detach", "FETCH_HEAD"],
               check=True, env=git_env)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO))
print("Code commit:", COMMIT)

### Install the recorded runtime

These are the versions used for the local reproduction checks. The original V5/V2 environments
have not been recovered yet; these pins do not establish exact reproduction of those submissions.

In [ ]:
if os.environ.get("FDS_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r",
                    str(REPO / "requirements-notebook.txt")], check=True)
print("Dependencies available")

## Data and execution controls

Attach `fds-pokemon-battles-prediction-2026` as the Kaggle competition data source.
The input folder must contain `train.jsonl`, `test.jsonl`, `type_chart_gen2.csv` and
`sample_submission.csv`. Inputs are read from their mount; generated files go under `OUTPUT_ROOT`.

All three approaches are selected by default. A subset is useful during development;
the final hand-in must generate all three. After changing the controls, rerun preflight
and the following cells. GitHub displays a static preview; controls run in Kaggle/Jupyter.

In [ ]:
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, FileLink
from submission_tools.runner import SUBMISSIONS, resolve_data, preflight, bundle

try:
    initial_data = str(resolve_data())
except FileNotFoundError:
    initial_data = os.environ.get("FDS_DATA", "")
labels = [("1 — Hidden knockouts + P1 history", SUBMISSIONS[0]),
          ("2 — Turn-31 forecast, CatBoost-dominant V5", SUBMISSIONS[1]),
          ("3 — Turn-31 forecast, stacked V2", SUBMISSIONS[2])]
default_selection = tuple(filter(None, os.environ.get(
    "FDS_SUBMISSIONS", ",".join(SUBMISSIONS)).split(",")))
selection = widgets.SelectMultiple(options=labels, value=default_selection,
                                   description="Approaches", rows=3,
                                   layout=widgets.Layout(width="650px"))
data_input = widgets.Text(value=initial_data, description="Data folder",
                          layout=widgets.Layout(width="850px"))
OUTPUT_ROOT = Path(os.environ.get("FDS_OUTPUT_ROOT", str(WORKSPACE / "final_submissions"))).resolve()
display(selection, data_input)
print("Outputs:", OUTPUT_ROOT)

## Checks — verify inputs and original source

The checks below verify original code hashes and report missing source before training starts.
The V2 check currently fails because its saved notebook depends on definitions from an old session.
Recover those original cells before running all three for the hand-in.

In [ ]:
selected = tuple(selection.value)
if not selected:
    raise ValueError("Select at least one approach")
DATA = resolve_data(data_input.value.strip() or None)
checks = [preflight(REPO / "submissions" / name, DATA) for name in selected]
results = {}
display(pd.DataFrame([{
    "Approach": check["submission"], "Ready to execute": check["ready_to_execute"],
    "Source issues": "; ".join(check["errors"]),
    "Reproduction note": check["reproduction_note"],
} for check in checks]))
if any(not check["ready_to_execute"] for check in checks):
    raise RuntimeError("A selected original source is incomplete. Recover it before the final Run All.")


def run_selected(name):
    if name not in selected:
        print("Skipped in this development run:", name)
        return
    results.pop(name, None)
    out = OUTPUT_ROOT / name
    env = dict(os.environ, FDS_DATA=str(DATA), MPLCONFIGDIR=str(WORKSPACE / "matplotlib_cache"))
    subprocess.run([sys.executable, "-u", str(REPO / "submissions" / name / "run.py"),
                    "--data", str(DATA), "--out", str(out)],
                   check=True, cwd=REPO, env=env)
    record = json.loads((out / "run_manifest.json").read_text())
    if record["git_commit"] != COMMIT:
        raise RuntimeError("The model ran from another code commit")
    results[name] = record
    print("Matches original submitted predictions:", record["reference"]["predictions_match"])
    display(FileLink(os.path.relpath(out / "submission.csv", Path.cwd())))

## Submission 1 — hidden knockouts and Player 1 history

A LightGBM model uses the state at turn 30, corrected for Player 2's hidden knockouts,
Player 1's species and items, move counts and an out-of-fold encoding of Player 1's team history.
The original seeds, model settings and repeated validation are preserved; extra ablation fits
are omitted when generating the final model. Reference submission: **57708941**.

In [ ]:
run_selected(SUBMISSIONS[0])

## Submission 2 — Turn-31 forecast, V5

The original pipeline predicts the next switches and active types, builds battle features
and combines CatBoost, XGBoost and LightGBM with its original 80/12/8 weights and 0.5 cutoff.
Only the final V5 cells run; the old V3/V4 experiments remain archived in the original notebook.
Reference submission: **57696039**. Exact prediction reproduction is still unresolved.

In [ ]:
run_selected(SUBMISSIONS[1])

## Submission 3 — Turn-31 forecast, V2

The original V2 pipeline uses a turn-31 forecast and a LightGBM/XGBoost/CatBoost stack,
followed by its original prior-calibrated threshold. Reference submission: **57675780**.
Its saved source is currently incomplete; missing definitions are listed in `source.json`.
The runner stops rather than reconstructing or substituting an algorithm.

In [ ]:
run_selected(SUBMISSIONS[2])

## Results and final export

A valid final run must generate all three CSVs from the same code commit and inputs.
The export additionally checks that every prediction fingerprint matches the corresponding
CSV actually submitted to Kaggle. Reference CSVs are not used for fitting or generation.

In [ ]:
if results:
    display(pd.DataFrame([{
        "Approach": name, "Rows": record["output"]["rows"],
        "Original submission": record["reference"]["submission_ref"],
        "Exact prediction match": record["reference"]["predictions_match"],
    } for name, record in results.items()]))
if set(results) != set(SUBMISSIONS):
    display(Markdown("**Development run only.** All three approaches are required for the final export."))
else:
    archive = bundle(list(results.values()), OUTPUT_ROOT)
    display(Markdown("All three prediction fingerprints match the original submissions."))
    display(FileLink(os.path.relpath(archive, Path.cwd())))

## Hand-in

Once the incomplete V2 source and V5 reproduction difference are resolved, leave all three
approaches selected and run the notebook from a fresh Kaggle session. Save a completed version
and make that Kaggle notebook public. The course form asks for the public Kaggle notebook and
report; the GitHub notebook is the versioned source, while the Kaggle copy is the runnable hand-in.

The detailed setup and hand-in steps are in `submissions/README.md` in the GitHub repository.